In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

tickers = ["TCS.NS", "HDFCBANK.NS", "RELIANCE.NS", "SUNPHARMA.NS", "ITC.NS"]

data = yf.download(tickers, start="2014-01-01", end="2023-12-31")["Close"] # Changed 'Adj Close' to 'Close'

returns = data.pct_change().dropna()

mu = returns.mean() * 252
sigma = returns.std() * np.sqrt(252)
cov = returns.cov() * 252

/tmp/ipykernel_822/119353555.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2014-01-01", end="2023-12-31")["Close"] # Changed 'Adj Close' to 'Close'
[*********************100%***********************]  5 of 5 completed


In [2]:
summary = pd.DataFrame({
    "Annualized Return": mu,
    "Annualized Risk": sigma
})

# Convert to percentage
summary["Annualized Return"] = summary["Annualized Return"] * 100
summary["Annualized Risk"] = summary["Annualized Risk"] * 100

# Round nicely
summary = summary.round(2)

print("\n=== Annualized Portfolio Statistics ===\n")
print(summary)

print("\n=== Annualized Covariance Matrix ===\n")
print(cov.round(4))


=== Annualized Portfolio Statistics ===

              Annualized Return  Annualized Risk
Ticker                                          
HDFCBANK.NS               20.01            22.46
ITC.NS                    14.24            25.35
RELIANCE.NS               23.49            27.84
SUNPHARMA.NS              13.18            29.78
TCS.NS                    17.94            24.03

=== Annualized Covariance Matrix ===

Ticker        HDFCBANK.NS  ITC.NS  RELIANCE.NS  SUNPHARMA.NS  TCS.NS
Ticker                                                              
HDFCBANK.NS        0.0505  0.0161       0.0254        0.0130  0.0124
ITC.NS             0.0161  0.0643       0.0162        0.0137  0.0107
RELIANCE.NS        0.0254  0.0162       0.0775        0.0186  0.0162
SUNPHARMA.NS       0.0130  0.0137       0.0186        0.0887  0.0125
TCS.NS             0.0124  0.0107       0.0162        0.0125  0.0577


In [4]:
# ----------------------------
# Step 5: Goal Sequences
# ----------------------------


sequence_A = {
    3: 1500000,
    7: 2500000,
    12: 3000000
}

sequence_B = {
    8: 1000000,
    12: 2000000,
    16: 4000000
}

terminal_goal = 15000000

**TASK 2.1**

In [5]:
from itertools import product
import numpy as np

weight_options = [0, 0.25, 0.5, 0.75, 1.0]

portfolios = []

for combo in product(weight_options, repeat=5):
    if abs(sum(combo) - 1.0) < 1e-8:
        portfolios.append(combo)

print("Total valid portfolios:", len(portfolios))

for p in portfolios:
    print(p)

# Test first portfolio
w = np.array(portfolios[0])

mu_p = np.dot(w, mu.values)
sigma_p = np.sqrt(w.T @ cov.values @ w)

print("Portfolio Return:", mu_p)
print("Portfolio Risk:", sigma_p)

Total valid portfolios: 70
(0, 0, 0, 0, 1.0)
(0, 0, 0, 0.25, 0.75)
(0, 0, 0, 0.5, 0.5)
(0, 0, 0, 0.75, 0.25)
(0, 0, 0, 1.0, 0)
(0, 0, 0.25, 0, 0.75)
(0, 0, 0.25, 0.25, 0.5)
(0, 0, 0.25, 0.5, 0.25)
(0, 0, 0.25, 0.75, 0)
(0, 0, 0.5, 0, 0.5)
(0, 0, 0.5, 0.25, 0.25)
(0, 0, 0.5, 0.5, 0)
(0, 0, 0.75, 0, 0.25)
(0, 0, 0.75, 0.25, 0)
(0, 0, 1.0, 0, 0)
(0, 0.25, 0, 0, 0.75)
(0, 0.25, 0, 0.25, 0.5)
(0, 0.25, 0, 0.5, 0.25)
(0, 0.25, 0, 0.75, 0)
(0, 0.25, 0.25, 0, 0.5)
(0, 0.25, 0.25, 0.25, 0.25)
(0, 0.25, 0.25, 0.5, 0)
(0, 0.25, 0.5, 0, 0.25)
(0, 0.25, 0.5, 0.25, 0)
(0, 0.25, 0.75, 0, 0)
(0, 0.5, 0, 0, 0.5)
(0, 0.5, 0, 0.25, 0.25)
(0, 0.5, 0, 0.5, 0)
(0, 0.5, 0.25, 0, 0.25)
(0, 0.5, 0.25, 0.25, 0)
(0, 0.5, 0.5, 0, 0)
(0, 0.75, 0, 0, 0.25)
(0, 0.75, 0, 0.25, 0)
(0, 0.75, 0.25, 0, 0)
(0, 1.0, 0, 0, 0)
(0.25, 0, 0, 0, 0.75)
(0.25, 0, 0, 0.25, 0.5)
(0.25, 0, 0, 0.5, 0.25)
(0.25, 0, 0, 0.75, 0)
(0.25, 0, 0.25, 0, 0.5)
(0.25, 0, 0.25, 0.25, 0.25)
(0.25, 0, 0.25, 0.5, 0)
(0.25, 0, 0.5, 0, 0.25)
(0.25, 0,

**Task 2.2: Monte Carlo Simulation**


In [6]:
def simulate_portfolio(mu_p, sigma_p, goals, n_paths=5000):
    success_count = 0
    terminal_goal = 15000000

    for _ in range(n_paths):
        wealth = 0
        loan = 0

        for year in range(1, 21):

            contribution = 240000 * (1.04)**(year - 1)
            wealth += contribution

            annual_return = np.random.normal(mu_p, sigma_p)

            wealth *= (1 + annual_return)

            loan *= 1.12

            if wealth >= loan:
                wealth -= loan
                loan = 0
            else:
                loan -= wealth
                wealth = 0

            if year in goals:
                target = goals[year]

                if wealth >= target:
                    wealth -= target
                else:
                    shortfall = target - wealth
                    loan += shortfall
                    wealth = 0

        if wealth >= terminal_goal:
            success_count += 1

    return success_count / n_paths

In [7]:
prob = simulate_portfolio(mu_p, sigma_p, sequence_A, n_paths=500)

print("Success Probability:", prob)

Success Probability: 0.004


In [9]:
results_A = []

for p in portfolios:
    w = np.array(p)

    mu_p = np.dot(w, mu.values)
    sigma_p = np.sqrt(w.T @ cov.values @ w)

    prob = simulate_portfolio(mu_p, sigma_p, sequence_A, n_paths=5000)

    results_A.append({
        "Weights": p,
        "Return": mu_p,
        "Risk": sigma_p,
        "Success Probability": prob
    })

df_A = pd.DataFrame(results_A)

df_A = df_A.sort_values(by="Success Probability", ascending=False)


print("Top 10 portfolios for sequence A")
print(df_A.head(10))

Top 10 portfolios for sequence A
                    Weights    Return      Risk  Success Probability
14        (0, 0, 1.0, 0, 0)  0.234910  0.278373               0.0546
44    (0.25, 0, 0.75, 0, 0)  0.226216  0.237208               0.0288
12    (0, 0, 0.75, 0, 0.25)  0.221035  0.230820               0.0218
60      (0.5, 0, 0.5, 0, 0)  0.217521  0.211395               0.0168
13    (0, 0, 0.75, 0.25, 0)  0.209137  0.236844               0.0148
24    (0, 0.25, 0.75, 0, 0)  0.211778  0.231711               0.0146
69        (1.0, 0, 0, 0, 0)  0.200131  0.224637               0.0096
67    (0.75, 0, 0.25, 0, 0)  0.208826  0.206767               0.0078
9       (0, 0, 0.5, 0, 0.5)  0.207159  0.204727               0.0066
42  (0.25, 0, 0.5, 0, 0.25)  0.212340  0.195168               0.0066


In [11]:
results_B = []

for p in portfolios:
    w = np.array(p)

    mu_p = np.dot(w, mu.values)
    sigma_p = np.sqrt(w.T @ cov.values @ w)

    prob = simulate_portfolio(mu_p, sigma_p, sequence_B, n_paths=5000)

    results_B.append({
        "Weights": p,
        "Return": mu_p,
        "Risk": sigma_p,
        "Success Probability": prob
    })

df_B = pd.DataFrame(results_B)

df_B = df_B.sort_values(by="Success Probability", ascending=False)

print("Top 10 portfolios for sequence B\n")
print(df_B.head(10))

Top 10 portfolios for sequence B

                    Weights    Return      Risk  Success Probability
42  (0.25, 0, 0.5, 0, 0.25)  0.212340  0.195168               0.8840
60      (0.5, 0, 0.5, 0, 0)  0.217521  0.211395               0.8760
58  (0.5, 0, 0.25, 0, 0.25)  0.203645  0.180411               0.8686
44    (0.25, 0, 0.75, 0, 0)  0.226216  0.237208               0.8526
12    (0, 0, 0.75, 0, 0.25)  0.221035  0.230820               0.8502
67    (0.75, 0, 0.25, 0, 0)  0.208826  0.206767               0.8398
39  (0.25, 0, 0.25, 0, 0.5)  0.198464  0.181005               0.8368
9       (0, 0, 0.5, 0, 0.5)  0.207159  0.204727               0.8342
50  (0.25, 0.25, 0.5, 0, 0)  0.203083  0.197391               0.8318
14        (0, 0, 1.0, 0, 0)  0.234910  0.278373               0.8232


**TASK 2.3 : Maximizing Success**

In [12]:
best_A = df_A.iloc[0]

print("=== Best Portfolio for Sequence A ===")
print(best_A)

=== Best Portfolio for Sequence A ===
Weights                (0, 0, 1.0, 0, 0)
Return                           0.23491
Risk                            0.278373
Success Probability               0.0546
Name: 14, dtype: object


In [13]:
best_B = df_B.iloc[0]

print("=== Best Portfolio for Sequence B ===")
print(best_B)

=== Best Portfolio for Sequence B ===
Weights                (0.25, 0, 0.5, 0, 0.25)
Return                                 0.21234
Risk                                  0.195168
Success Probability                      0.884
Name: 42, dtype: object


BONUS

In [18]:
# ==========================================
# BONUS PART: Continuous Portfolio Optimization
# ==========================================

from scipy.optimize import minimize
import numpy as np
import pandas as pd

# --------------------------------------------------
# Monte Carlo Simulation Function
# --------------------------------------------------
def simulate_portfolio(weights, mu, cov, goals, terminal_goal,
                       n_paths=3000, years=20):

    # Portfolio expected return and volatility
    mu_p = np.dot(weights, mu.values)
    sigma_p = np.sqrt(weights.T @ cov.values @ weights)

    success_count = 0

    for _ in range(n_paths):
        wealth = 0
        loan = 0
        monthly_saving = 20000

        for year in range(1, years + 1):

            # Add yearly savings
            annual_saving = monthly_saving * 12
            wealth += annual_saving

            # Simulated annual portfolio return
            r = np.random.normal(mu_p, sigma_p)

            # Prevent impossible returns below -100%
            r = max(r, -0.99)

            wealth *= (1 + r)

            # Repay existing loan first
            if loan > 0:
                repayment = min(wealth, loan)
                wealth -= repayment
                loan -= repayment

            # Intermediate goals only
            if year in goals:
                target = goals[year]

                if wealth >= target:
                    wealth -= target
                else:
                    shortfall = target - wealth
                    wealth = 0
                    loan += shortfall * 1.12   # 12% borrowing penalty

            # Savings grow annually by 4%
            monthly_saving *= 1.04

        # Success only if:
        # 1) no remaining debt
        # 2) retirement target achieved
        if loan == 0 and wealth >= terminal_goal:
            success_count += 1

    return success_count / n_paths


# --------------------------------------------------
# Objective Function
# --------------------------------------------------
def objective(weights, mu, cov, goals, terminal_goal):
    """
    Negative success probability for minimization.
    """
    np.random.seed(42)   # stabilize optimization
    prob = simulate_portfolio(weights, mu, cov, goals, terminal_goal)
    return -prob


# --------------------------------------------------
# Constraints and Bounds
# --------------------------------------------------

# Weights must sum to 1
constraints = ({
    'type': 'eq',
    'fun': lambda w: np.sum(w) - 1
})

# Allow short-selling / leverage
bounds = [(-0.5, 1.5) for _ in range(len(tickers))]

# Initial guess: equal allocation
initial_weights = np.array([1/len(tickers)] * len(tickers))


# --------------------------------------------------
# Define Goal Sequences
# --------------------------------------------------

# Sequence A
goals_A = {
    3: 1500000,
    7: 2500000,
    12: 3000000
}
terminal_A = 15000000

# Sequence B
goals_B = {
    8: 1000000,
    12: 2000000,
    16: 4000000
}
terminal_B = 15000000


# --------------------------------------------------
# Run Optimization
# --------------------------------------------------

result_A = minimize(
    objective,
    initial_weights,
    args=(mu, cov, goals_A, terminal_A),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

result_B = minimize(
    objective,
    initial_weights,
    args=(mu, cov, goals_B, terminal_B),
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)


# --------------------------------------------------
# Display Results
# --------------------------------------------------

def display_results(result, label):
    weights = result.x
    success_prob = -result.fun
    mu_p = np.dot(weights, mu.values)
    sigma_p = np.sqrt(weights.T @ cov.values @ weights)

    print(f"\n===== {label} =====")
    print("Optimal Weights:")
    for t, w in zip(tickers, weights):
        print(f"{t}: {w:.4f}")

    print(f"\nPortfolio Expected Return: {mu_p:.4f}")
    print(f"Portfolio Risk: {sigma_p:.4f}")
    print(f"Max Success Probability: {success_prob:.4f}")


display_results(result_A, "Sequence A")
display_results(result_B, "Sequence B")


===== Sequence A =====
Optimal Weights:
TCS.NS: 0.2000
HDFCBANK.NS: 0.2000
RELIANCE.NS: 0.2000
SUNPHARMA.NS: 0.2000
ITC.NS: 0.2000

Portfolio Expected Return: 0.1777
Portfolio Risk: 0.1610
Max Success Probability: 0.0003

===== Sequence B =====
Optimal Weights:
TCS.NS: 0.2000
HDFCBANK.NS: 0.2000
RELIANCE.NS: 0.2000
SUNPHARMA.NS: 0.2000
ITC.NS: 0.2000

Portfolio Expected Return: 0.1777
Portfolio Risk: 0.1610
Max Success Probability: 0.7410


In [19]:
# ==========================================
# BONUS UPGRADE: Differential Evolution
# ==========================================

from scipy.optimize import differential_evolution
import numpy as np

# --------------------------------------------------
# Monte Carlo Simulation
# --------------------------------------------------
def simulate_portfolio(weights, mu, cov, goals, terminal_goal,
                       n_paths=3000, years=20):

    mu_p = np.dot(weights, mu.values)
    sigma_p = np.sqrt(weights.T @ cov.values @ weights)

    success_count = 0

    for _ in range(n_paths):
        wealth = 0
        loan = 0
        monthly_saving = 20000

        for year in range(1, years + 1):

            # yearly savings
            annual_saving = monthly_saving * 12
            wealth += annual_saving

            # simulate annual return
            r = np.random.normal(mu_p, sigma_p)
            r = max(r, -0.99)

            wealth *= (1 + r)

            # repay loans
            if loan > 0:
                repayment = min(wealth, loan)
                wealth -= repayment
                loan -= repayment

            # intermediate goals
            if year in goals:
                target = goals[year]

                if wealth >= target:
                    wealth -= target
                else:
                    shortfall = target - wealth
                    wealth = 0
                    loan += shortfall * 1.12

            # savings growth
            monthly_saving *= 1.04

        # terminal success
        if loan == 0 and wealth >= terminal_goal:
            success_count += 1

    return success_count / n_paths


# --------------------------------------------------
# Penalized Objective Function
# --------------------------------------------------
def objective_de(weights, mu, cov, goals, terminal_goal):

    # penalty if weights don't sum to 1
    penalty = 1000 * abs(np.sum(weights) - 1)

    np.random.seed(42)

    prob = simulate_portfolio(weights, mu, cov, goals, terminal_goal)

    return -prob + penalty


# --------------------------------------------------
# Optimization Setup
# --------------------------------------------------

bounds = [(-0.5, 1.5) for _ in range(len(tickers))]

# Sequence A
goals_A = {
    3: 1500000,
    7: 2500000,
    12: 3000000
}
terminal_A = 15000000

# Sequence B
goals_B = {
    8: 1000000,
    12: 2000000,
    16: 4000000
}
terminal_B = 15000000


# --------------------------------------------------
# Run Differential Evolution
# --------------------------------------------------

result_A = differential_evolution(
    objective_de,
    bounds=bounds,
    args=(mu, cov, goals_A, terminal_A),
    strategy="best1bin",
    maxiter=40,
    popsize=15,
    tol=0.01,
    seed=42
)

result_B = differential_evolution(
    objective_de,
    bounds=bounds,
    args=(mu, cov, goals_B, terminal_B),
    strategy="best1bin",
    maxiter=40,
    popsize=15,
    tol=0.01,
    seed=42
)


# --------------------------------------------------
# Normalize Weights to Sum = 1
# --------------------------------------------------
def normalize_weights(w):
    return w / np.sum(w)


# --------------------------------------------------
# Display Results
# --------------------------------------------------
def display_results(result, label, goals, terminal_goal):

    weights = normalize_weights(result.x)

    success_prob = simulate_portfolio(
        weights, mu, cov, goals, terminal_goal, n_paths=5000
    )

    mu_p = np.dot(weights, mu.values)
    sigma_p = np.sqrt(weights.T @ cov.values @ weights)

    print(f"\n===== {label} =====")
    print("Optimal Weights:")
    for t, w in zip(tickers, weights):
        print(f"{t}: {w:.4f}")

    print(f"\nPortfolio Expected Return: {mu_p:.4f}")
    print(f"Portfolio Risk: {sigma_p:.4f}")
    print(f"Max Success Probability: {success_prob:.4f}")


display_results(result_A, "Sequence A", goals_A, terminal_A)
display_results(result_B, "Sequence B", goals_B, terminal_B)


===== Sequence A =====
Optimal Weights:
TCS.NS: -0.4492
HDFCBANK.NS: -0.0823
RELIANCE.NS: 1.1802
SUNPHARMA.NS: 0.2132
ITC.NS: 0.1381

Portfolio Expected Return: 0.2285
Portfolio Risk: 0.3247
Max Success Probability: 0.0644

===== Sequence B =====
Optimal Weights:
TCS.NS: -0.1273
HDFCBANK.NS: -0.2508
RELIANCE.NS: 0.6831
SUNPHARMA.NS: 0.6141
ITC.NS: 0.0809

Portfolio Expected Return: 0.1947
Portfolio Risk: 0.2785
Max Success Probability: 0.6104
